In [1]:
import os
import pandas as pd
import cv2
import numpy as np

In [ ]:
DATA_DIR = '/Volumes/Dados/tcc/SMIC_all_cropped'
from pathlib import Path
# OUTPUT_DIR deve ser um Path para usar operador / e garantir existência
OUTPUT_DIR = Path('../../data/SMIC/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'OUTPUT_DIR definido em: {OUTPUT_DIR.resolve()}')

In [22]:
from pathlib import Path
DATA_DIR = Path('/Volumes/Dados/tcc/SMIC_all_cropped')
subjects = [p for p in DATA_DIR.glob('**/') if p.is_dir()]
print(f'Total de pastas encontradas: {len(subjects)}')

Total de pastas encontradas: 808


In [19]:
# Benchmark rápido: mede tempo médio de leitura+resize em uma amostra e estima tempo total
import time
from itertools import islice
SAMPLE_N = 200
# encontrar até SAMPLE_N arquivos válidos (ignora arquivos que começam com ._)
paths = []
for subj in subjects:
    for f in subj.glob("**/*.bmp"):
        if not f.name.startswith("._") and f.is_file():
            paths.append(f)
        if len(paths) >= SAMPLE_N:
            break
    if len(paths) >= SAMPLE_N:
        break
print(f"Amostra coletada: {len(paths)} arquivos")
if not paths:
    print('Nenhuma imagem .bmp válida encontrada para amostra.')
else:
    t0 = time.perf_counter()
    valid = 0
    for p in paths:
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (112, 112))
        valid += 1
    t1 = time.perf_counter()
    if valid == 0:
        print('Nenhuma imagem válida lida na amostra.')
    else:
        per_img = (t1 - t0) / valid
        # contar total de .bmp válidos (ignora ._*)
        total_bmps = sum(1 for f in DATA_DIR.glob('**/*.bmp') if not f.name.startswith('._') and f.is_file())
        est_total_s = per_img * total_bmps
        print(f"Tempo médio por imagem (leitura+resize): {per_img:.4f} s")
        print(f"Total .bmp válidos encontrados: {total_bmps}")
        print(f"Estimativa total: {est_total_s/60:.1f} minutos ({est_total_s:.0f} segundos)")
        # mostrar também estimativa por pasta média
        avg_frames_per_subj = total_bmps / max(1, len(subjects))
        print(f"Frames médios por pasta: {avg_frames_per_subj:.1f}")

Amostra coletada: 200 arquivos
Tempo médio por imagem (leitura+resize): 0.0110 s
Total .bmp válidos encontrados: 13296
Estimativa total: 2.4 minutos (146 segundos)
Frames médios por pasta: 16.5


In [23]:
# Processamento paralelo (ThreadPool) — lê e redimensiona imagens em paralelo e reconstrói sequências por pasta
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
num_workers = min(8, (os.cpu_count() or 1) * 2)
print(f'Usando {num_workers} workers para I/O/CPU-bound mix')
# coletar todos os caminhos válidos (ignora arquivos '._')
all_paths = [f for f in DATA_DIR.glob('**/*.bmp') if not f.name.startswith('._') and f.is_file()]
print(f'Total imagens válidas encontradas: {len(all_paths)}')
if not all_paths:
    print('Nenhuma imagem .bmp válida encontrada — verifique o caminho')
else:
    def read_resize(path):
        try:
            img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                return (path, None)
            img = cv2.resize(img, (112, 112))
            return (path, img)
        except Exception as e:
            return (path, None)
    t0 = time.perf_counter()
    results = {}  # path -> img array
    with ThreadPoolExecutor(max_workers=num_workers) as ex:
        futures = {ex.submit(read_resize, p): p for p in all_paths}
        for i, fut in enumerate(as_completed(futures), 1):
            p, img = fut.result()
            if img is not None:
                results[p] = img
            else:
                # opcional: registrar imagens problemáticas
                pass
            if i % 1000 == 0:
                print(f'Processados {i}/{len(all_paths)}')
    t1 = time.perf_counter()
    successful = len(results)
    print(f'Leitura+resize paralelos concluídos: {successful}/{len(all_paths)} imagens válidas lidas em {t1-t0:.1f}s')
    per_img = (t1 - t0) / max(1, successful)
    print(f'Tempo médio por imagem (paralelo): {per_img:.4f} s')
    # reagrupar por pasta de vídeo e reconstruir sequências ordenadas
    grouped = defaultdict(list)
    for p, img in results.items():
        grouped[p.parent].append((p.name, img))
    frames_all = []
    labels_all = []
    for subj_path, items in grouped.items():
        # ordenar por nome de arquivo para manter sequência temporal
        items.sort(key=lambda x: x[0])
        seq = [img for _, img in items]
        if not seq:
            continue
        seq = np.stack(seq)
        frames_all.append(seq)
        # mesma lógica anterior: rótulo = pasta pai do diretório do vídeo (ex: 'negative')
        labels_all.append(subj_path.parent.name)
    print(f'Total de sequências reconstruídas: {len(frames_all)}')

Usando 8 workers para I/O/CPU-bound mix
Total imagens válidas encontradas: 13296
Total imagens válidas encontradas: 13296
Processados 1000/13296
Processados 1000/13296
Processados 2000/13296
Processados 2000/13296
Processados 3000/13296
Processados 3000/13296
Processados 4000/13296
Processados 4000/13296
Processados 5000/13296
Processados 5000/13296
Processados 6000/13296
Processados 6000/13296
Processados 7000/13296
Processados 7000/13296
Processados 8000/13296
Processados 8000/13296
Processados 9000/13296
Processados 9000/13296
Processados 10000/13296
Processados 10000/13296
Processados 11000/13296
Processados 11000/13296
Processados 12000/13296
Processados 12000/13296
Processados 13000/13296
Processados 13000/13296
Leitura+resize paralelos concluídos: 13296/13296 imagens válidas lidas em 146.1s
Tempo médio por imagem (paralelo): 0.0110 s
Total de sequências reconstruídas: 612
Leitura+resize paralelos concluídos: 13296/13296 imagens válidas lidas em 146.1s
Tempo médio por imagem (par

In [29]:
# Codificar os labels (strings) em inteiros para uso em modelos
from sklearn.preprocessing import LabelEncoder
import json
from pathlib import Path
le = LabelEncoder()
# Garantir que OUTPUT_DIR seja um Path (proteção se células foram executadas fora de ordem)
if not isinstance(OUTPUT_DIR, Path):
    OUTPUT_DIR = Path(str(OUTPUT_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Verificar se frames_all / labels_all foram preenchidos antes de prosseguir
if not frames_all or not labels_all:
    print('Nenhuma sequência/processamento disponível. Execute a célula de processamento antes de codificar e salvar.')
else:
    # codificar labels
    y_encoded = le.fit_transform(labels_all)
    print('Labels originais:', set(labels_all))
    print('Labels codificados:', set(y_encoded))
    # Pad/Truncate sequences para comprimento uniforme antes de criar um ndarray
    # escolher max_frames: aqui usamos 16 (pode ajustar conforme necessário)
    max_frames = 16
    padded = []
    for seq in frames_all:
        if seq.shape[0] < max_frames:
            pad = np.zeros((max_frames - seq.shape[0], seq.shape[1], seq.shape[2]), dtype=seq.dtype)
            new_seq = np.vstack([seq, pad])
        else:
            new_seq = seq[:max_frames]
        padded.append(new_seq)
    X = np.array(padded)   # shape: (N_vídeos, T, H, W)
    y = np.array(y_encoded)    # shape: (N_vídeos,)
    print('Shape X:', X.shape, 'Shape y:', y.shape)
    # salvar arrays e mapping
    np.save(OUTPUT_DIR / 'X.npy', X)
    np.save(OUTPUT_DIR / 'y.npy', y)
    # converter numpy types para int nativo antes de serializar
    mapping = {str(cls): int(val) for cls, val in zip(le.classes_, le.transform(le.classes_))}
    with open(OUTPUT_DIR / 'label_mapping.json', 'w', encoding='utf-8') as f:
        json.dump(mapping, f, indent=2, ensure_ascii=False)
    # também salvar X_padded separadamente por compatibilidade com células posteriores
    np.save(OUTPUT_DIR / 'X_padded.npy', X)


Labels originais: {'positive', 'negative', 'surprise', 'non_micro'}
Labels codificados: {0, 1, 2, 3}
Shape X: (612, 16, 112, 112) Shape y: (612,)


In [30]:
def pad_sequences(sequences, max_len):
    padded = []
    for seq in sequences:
        if seq.shape[0] < max_len:
            # padding com zeros
            pad = np.zeros((max_len - seq.shape[0], seq.shape[1], seq.shape[2]))
            new_seq = np.vstack([seq, pad])
        else:
            new_seq = seq[:max_len]
        padded.append(new_seq)
    return np.array(padded)

max_frames = 16
X_padded = pad_sequences(frames_all, max_frames)
np.save(OUTPUT_DIR / "X_padded.npy", X_padded)
print("X_padded:", X_padded.shape)

X_padded: (612, 16, 112, 112)


In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y, test_size=0.3, random_state=42, stratify=y
)

np.save(OUTPUT_DIR / "X_train.npy", X_train)
np.save(OUTPUT_DIR / "X_test.npy", X_test)
np.save(OUTPUT_DIR / "y_train.npy", y_train)
np.save(OUTPUT_DIR / "y_test.npy", y_test)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (428, 16, 112, 112) (428,)
Test : (184, 16, 112, 112) (184,)
